## U24AI040 NLP LAB 2 ## 

You are given a dataset in JSON format text_segmentation_dataset.json. The dataset is a
snapshot from the large Brown corpus. You need to implement two text segmentation
techniques to segment the text into words and report their performance.
1. Greedy Based Approach that matches the longest word
2. Dynamic Programming Approach that increases the log probability of the text [Hint:
Frequencies of Words are given]
You need to report two evaluation metrics.
1. Accuracy
2. Edit Distance

In [1]:
import json
import math

In [2]:
with open("text_segmentation_dataset.json","r",encoding="utf-8") as f :
  data=json.load(f)

In [3]:
word_count=data["word_counts"] # Dictionary of Word and it frequency
total_unique_words=len(word_count) # Size of Dictionary

In [4]:
v_size=data["metadata"]["vocabulary_size"] 

total_words=data["metadata"]["total_corpus_words"] #Total Words(including duplicates)

no_inputs=data["metadata"]["test_case_count"] # no of test case
print(no_inputs)

1000


In [5]:
test_cases=data["test_cases"]
print(test_cases)

[{'input': 'itthatthecitytakestepstothisproblem', 'ground_truth': 'it that the city take steps to this problem', 'word_count': 9}, {'input': 'oftitlelawwasalsobythe', 'ground_truth': 'of title law was also by the', 'word_count': 7}, {'input': 'failuretodothiswillcontinuetoplaceaon', 'ground_truth': 'failure to do this will continue to place a on', 'word_count': 10}, {'input': 'onotherthethat', 'ground_truth': 'on other the that', 'word_count': 4}, {'input': 'williamforfromhiswifeincourt', 'ground_truth': 'william for from his wife in court', 'word_count': 7}, {'input': 'thecouplewasmarried', 'ground_truth': 'the couple was married', 'word_count': 4}, {'input': 'theyhaveasonwilliamandadaughterof', 'ground_truth': 'they have a son william and a daughter of', 'word_count': 9}, {'input': 'forthesaidthatanpropertyhasbeenagreedupon', 'ground_truth': 'for the said that an property has been agreed upon', 'word_count': 10}, {'input': 'thetheasandhisageas', 'ground_truth': 'the the as and his ag

In [6]:
def greedy_method(test,vocabulary): #Greedy Method
  text=test["input"]
  res=[]
  i=0
  n=len(text)
  while i<n :
    l_word=None

    for k in range(i+1,n+1):
      word=text[i:k]
      if word in vocabulary:
        if l_word is None or len(word)>len(l_word):
          l_word=word

    if l_word is not None:
      res.append(l_word)
      i+=len(l_word)
    else :
      res.append(text[i])
      i+=1
  return res

In [ ]:
def dp_method(test,vocabulary): #Dp Method
  text=test["input"]
  n=len(text)
  dp=[]
  prev=[]

  for i in range(0,n+1):
    dp.append(-float("inf"))
    prev.append(-1)

  dp[0]=0

  for i in range(n):
    if dp[i]==-float("inf"):
      continue

    for j in range(i+1,n+1):
      curr=text[i:j]
      if curr in vocabulary:
        p=(vocabulary[curr]/total_words)
        lp=math.log(p)
        score=dp[i]+lp
        if score>dp[j]:
          dp[j]=score
          prev[j]=i

  res=[]
  ind=n
  while ind > 0:
    i=prev[ind]
    if i==-1 :
      return None
    word=text[i:ind]
    res.append(word)
    ind =i
  res.reverse()
  return res


In [8]:
def accuracy_score(actual, predicted):
  sen=0
  correct=0
  l=len(predicted)
  t=len(actual)
  k=min(l,t)
  for i in range(k):
    if actual[i]==predicted[i]:
      correct+=1
  acc=correct/t
  if l==t and acc==1:
    sen=1 # Sentence is correctly predicted by algo

  return correct,sen,acc


In [9]:
# curr   = predicted
# actual = ground truth

# dp[i][j] = minimum edits to convert first i predicted words into first j actual words
def solve(i,j,actual,curr,dp):
  if i==0 and j==0:                   
    return 0
  if i==0:
    return j #insert

  if j==0:
    return i # delete
  if dp[i][j]!=-1:
    return dp[i][j]

  
  if actual[j-1]==curr[i-1]:
    dp[i][j]=solve(i-1,j-1,actual,curr,dp)
  else:

    delete=solve(i-1,j,actual,curr,dp)+1
    update=solve(i-1,j-1,actual,curr,dp)+1
    add=solve(i,j-1,actual,curr,dp)+1
    dp[i][j]=min(delete,update,add)

  return dp[i][j]


In [10]:
greedy_accuracy=0
greedy_correct_word=0
greedy_correct_sentence=0

dp_accuracy=0
dp_correct_word=0
dp_correct_sentence=0

correct_words_in_test_case=0

greedy_edit_distance = 0
dp_edit_distance = 0


In [ ]:
for test in test_cases:
  
  actual=test["ground_truth"].split()
  correct_words_in_test_case+=len(actual)
  
  g_predicted=greedy_method(test,word_count)
  g_corr,g_sen,g_acc=accuracy_score(actual,g_predicted)
  greedy_correct_word+=g_corr
  greedy_accuracy+=g_acc
  greedy_correct_sentence+=g_sen

  d_predicted=dp_method(test,word_count)
  dp_corr,dp_sen,dp_acc=accuracy_score(actual,d_predicted)
  dp_correct_word+=dp_corr
  dp_accuracy+=dp_acc
  dp_correct_sentence+=dp_sen

# Edit Distance for Greedy Method
  dp = [[-1] * (len(actual) + 1)
        for _ in range(len(g_predicted) + 1)
    ]
  
  g_dis=solve(len(g_predicted),len(actual),actual,g_predicted,dp)
  greedy_edit_distance+=g_dis

# Edit Distance for DP Method
  dp = [[-1] * (len(actual) + 1)
      for _ in range(len(d_predicted) + 1)
  ]
  dp_dis=solve(len(d_predicted),len(actual),actual,d_predicted,dp)
  dp_edit_distance+=dp_dis
  


print("Accuracy for Greedy Method: ")
print("Total Correct word predicted by greedy method : ",greedy_correct_word)
print("Accuracy by word : ",round((greedy_correct_word/correct_words_in_test_case),5))
print("Accuracy by sentence : ",round(greedy_correct_sentence/no_inputs,5))
print("Average Accuracy in each sentence : ",greedy_accuracy/len(test_cases))
print()
print("Accuracy for Dp Method: ")
print("Total Correct word predicted by DP method : ",dp_correct_word)
print("Accuracy by word : ",round(dp_correct_word/correct_words_in_test_case,3))
print("Accuracy by sentence : ",round(dp_correct_sentence/no_inputs,3))
print("Average Accuracy in each sentence : ",dp_accuracy/len(test_cases))

print()
print("Edit Distance for Greedy Method : ")
print("Total Edit Distance:",greedy_edit_distance)
print("Average Edit Distance:",greedy_edit_distance / len(test_cases))

print()
print("Edit Distance for Dp Method : ")
print("Total Edit Distance:",dp_edit_distance)
print("Average Edit Distance:",dp_edit_distance / len(test_cases))

print("Greedy Edit Distance:", greedy_edit_distance)
print("DP Edit Distance:", dp_edit_distance)

print("Greedy incorrect sentences:",
      no_inputs - greedy_correct_sentence)

print("DP incorrect sentences:",
      no_inputs - dp_correct_sentence)
  
  


Accuracy for Greedy Method: 
Total Correct word predicted by greedy method :  6849
Accuracy by word :  0.82211
Accuracy by sentence :  0.691
Average Accuracy in each sentence :  0.8343584054834053

Accuracy for Dp Method: 
Total Correct word predicted by DP method :  8259
Accuracy by word :  0.991
Accuracy by sentence :  0.982
Average Accuracy in each sentence :  0.9903365079365078

Edit Distance for Greedy Method : 
Total Edit Distance: 1260
Average Edit Distance: 1.26

Edit Distance for Dp Method : 
Total Edit Distance: 37
Average Edit Distance: 0.037
Greedy Edit Distance: 1260
DP Edit Distance: 37
Greedy incorrect sentences: 309
DP incorrect sentences: 18


In [ ]:
# Sample Output
for i in range(10):

    test=test_cases[i]

    actual=test["ground_truth"].split()
    greedy_result=greedy_method(test,word_count)

    dp_result=dp_method(test,word_count)

    print("Test Case:",i+1)

    print("Input:")
    print(test["input"])

    print("\nGround Truth:")
    print(" ".join(actual))

    print("\nGreedy Segmentation:")
    print(" ".join(greedy_result))

    print("\nDP Segmentation:")
    print(" ".join(dp_result))

    print()

Test Case: 1
Input:
itthatthecitytakestepstothisproblem

Ground Truth:
it that the city take steps to this problem

Greedy Segmentation:
it that the city takes t e p s to this problem

DP Segmentation:
it that the city take steps to this problem

Test Case: 2
Input:
oftitlelawwasalsobythe

Ground Truth:
of title law was also by the

Greedy Segmentation:
of title law was also by the

DP Segmentation:
of title law was also by the

Test Case: 3
Input:
failuretodothiswillcontinuetoplaceaon

Ground Truth:
failure to do this will continue to place a on

Greedy Segmentation:
failure to do this will continue top l a c e a on

DP Segmentation:
failure to do this will continue to place a on

Test Case: 4
Input:
onotherthethat

Ground Truth:
on other the that

Greedy Segmentation:
on other the that

DP Segmentation:
on other the that

Test Case: 5
Input:
williamforfromhiswifeincourt

Ground Truth:
william for from his wife in court

Greedy Segmentation:
william for from his wife in court

DP Segm